# 01 - Data sources and exploratory analysis

Rubric: *Data sources (4)* and *Transformations + EDA (3)*. The data comes from four sources, stored as
parquet in `data/raw/` (gitignored, rebuilt by `--mode live`):

| Source | What | Module |
| --- | --- | --- |
| Yahoo Finance screener | every Thai common stock on SET / mai (universe discovery) | `ingest.discover_universe` |
| Yahoo Finance prices / profiles | daily OHLCV for ~850 stocks since 2010, global market series, sectors | `ingest.ingest_prices`, `ingest_market`, `ingest_sectors` |
| FRED | US 10y / 3m yields, Baa credit spread, THB/USD | `ingest.ingest_fred` |
| **Cboe CSV feed (new source)** | VIX9D, VIX, VIX3M term structure, VVIX, SKEW | `ingest.ingest_cboe` |

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
from setalpha import config, ingest, transform, model, regime, portfolio, strategy, evaluate, report, pipeline
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 180)
pd.set_option("display.float_format", "{:.4f}".format)
plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True})

In [ ]:
raw = ingest.ingest_all("file")
pd.DataFrame({k: {"rows": len(v), "columns": v.shape[1],
                  "from": v["Date"].min().date() if "Date" in v else "", "to": v["Date"].max().date() if "Date" in v else ""}
              for k, v in raw.items() if v is not None}).T

## Universe and the combined dataset

In [ ]:
df = transform.load()
print(f"combined dataset: {len(df):,} rows x {df.shape[1]} columns  |  {df['Ticker'].nunique()} stocks  |  "
      f"{df['Date'].min().date()} -> {df['Date'].max().date()}")
yearly = df.groupby(df["Date"].dt.year).agg(stocks=("Ticker", "nunique"), rows=("Ticker", "size"),
                                            liquid=("liquid", "sum"), tradable=("tradable", "sum"))
yearly

In [ ]:
df.loc[df["tradable"], "sector"].value_counts(normalize=True).mul(100).round(1).plot.barh(title="Tradable rows by sector (%)", figsize=(8, 4));

## What we predict (TO_PREDICT)

* `fwd_ret_5` - return from the next open to the close 5 days later (what a trade earns)
* `fwd_alpha_5` - the same minus beta x market: the stock-specific part (**main regression target**)
* `is_alpha_5` - 1 if `fwd_alpha_5 > 0` (**main classification target**); `is_positive_5` - 1 if `fwd_ret_5 > 0`

In [ ]:
t = df[df["tradable"]]
print(t[["fwd_ret_5", "fwd_alpha_5"]].describe(percentiles=[.01, .05, .5, .95, .99]).T)
print("share of positive alpha:", round(t["is_alpha_5"].mean(), 3), "| share of positive return:", round(t["is_positive_5"].mean(), 3))
t["fwd_alpha_5"].clip(-0.2, 0.2).hist(bins=120, figsize=(10, 3)); plt.title("5-day forward alpha, tradable names (clipped at +-20%)");

## Correlation of features with the target

Spearman rank correlation of each numeric feature with `fwd_alpha_5` on the training years (2010-2019) only,
so the choice of features does not peek at the test period.

In [ ]:
sets = transform.feature_sets(df)
train = t[t["Date"] < config.VALID_START]
corr = train[sets["NUMERIC"]].corrwith(train["fwd_alpha_5"], method="spearman").sort_values()
fig, ax = plt.subplots(figsize=(8, 12)); corr.plot.barh(ax=ax, title="Spearman corr. with 5-day forward alpha (train years)");
print("strongest negative:", corr.head(5).round(3).to_dict()); print("strongest positive:", corr.tail(5).round(3).to_dict())

In [ ]:
top = corr.abs().sort_values(ascending=False).head(12).index
fig, ax = plt.subplots(figsize=(9, 7)); im = ax.imshow(train[top].corr(method="spearman"), cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(top)), top, rotation=90); ax.set_yticks(range(len(top)), top); plt.colorbar(im); ax.set_title("Feature-feature correlation (top 12)");

## Market regimes: Gaussian HMM

Three hidden states fitted on [market return, log 20-day volatility]. Probabilities shown are *filtered* (no future
information) with yearly walk-forward refits.

In [ ]:
mkt = df[df["liquid"]].groupby("Date")["ret_1"].mean()
obs = regime.observations(mkt)
hmm = regime.fit_hmm(obs[obs.index < config.TEST_START])
print(regime.describe(hmm, obs).round(3))
reg = df.groupby("Date")[["hmm_p_stressed"]].first()
bench = raw["market"].query("Ticker == @config.BENCHMARK").set_index("Date")["Close"]
fig, ax = plt.subplots(figsize=(13, 4)); ax.fill_between(reg.index, reg["hmm_p_stressed"], alpha=.35, color="red", label="P(stressed)")
ax2 = ax.twinx(); ax2.plot(bench.index, bench, color="black", lw=.8, label="TDEX (SET50 ETF)"); ax.set_title("Filtered P(stressed) vs SET50 ETF"); ax.legend(loc="upper left");

In [ ]:
# does the regime matter for the target? mean 5-day market return by regime (training years)
mk = df[df["liquid"] & (df["Date"] < config.VALID_START)].groupby(["Date", "hmm_state"])["fwd_ret_5"].mean().reset_index()
mk.groupby("hmm_state")["fwd_ret_5"].agg(["mean", "std", "count"]).mul([100, 100, 1])